# 09 - Review of the Updated WoE / IV Analysis and Feature-Selection Methodology

**Project:** German Credit Risk
**Inputs:** raw data and the final files from notebook 08 (`german_credit_woe_final.csv`, `final_woe_iv_table.csv`)
**Author:** Divya Akbari

**Goal of this notebook**
1. **Review the updated WoE / IV analysis** (the final bands adopted in notebook 08).
2. **Validate the feature calculations** with rule-by-rule and boundary tests.
3. **Check the feature-selection methodology**: is the selection stable, unbiased, robust to its settings, consistent with other methods, free of leakage and fair?
4. Give the **final recommendations**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import chi2_contingency, spearmanr
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, cross_val_score
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')   # hide library version notices

pd.set_option('display.width', 150)
pd.set_option('display.max_colwidth', None)

raw = pd.read_csv('data/german_credit_data.csv').drop(columns=['Unnamed: 0'])
fe = pd.read_csv('data/processed/german_credit_engineered.csv')
final_woe = pd.read_csv('data/processed/german_credit_woe_final.csv')
final_table = pd.read_csv('data/processed/final_woe_iv_table.csv')

fe['housing'] = fe['Housing']
fe['sex'] = fe['Sex']
fe['home_owner'] = np.where(fe['housing'] == 'own', 'own', 'rent/free')
fe['purpose_type'] = np.where(fe['purpose_group'] == 'radio/TV', 'radio/TV', 'other')

final = ['checking_status', 'duration_bin', 'saving_status', 'credit_bin', 'payment_bin', 'age_bin', 'home_owner', 'purpose_type']
names = {'checking_status': 'Checking account status', 'low_balance_profile': 'Low balance profile', 'duration_bin': 'Duration',
         'saving_status': 'Saving account status', 'credit_bin': 'Credit amount', 'payment_bin': 'Monthly payment', 'age_bin': 'Age',
         'home_owner': 'Housing (own vs rent/free)', 'purpose_type': 'Purpose (radio/TV vs other)', 'sex': 'Sex', 'job_group': 'Job group'}

def woe_iv(data, f):
    g = data.groupby(f)['bad'].agg(n='size', b='sum')
    g['g'] = g['n'] - g['b']
    G, B = (data['bad'] == 0).sum(), data['bad'].sum()
    woe = np.log(((g['g'] + 0.5) / G) / ((g['b'] + 0.5) / B))
    return woe, (((g['g'] + 0.5) / G - (g['b'] + 0.5) / B) * woe).sum()
print(fe.shape, final_woe.shape, final_table.shape)

(1000, 30) (1000, 9) (26, 8)


## 1. Review of the updated WoE / IV analysis

The final bands must follow the binning rules: every band holds at least 5% of customers and has both good and bad customers.
Ordered features should show a logical risk pattern.

In [2]:
ordered = {'duration_bin': ['up to 11', '12-23', '24-35', '36+'], 'age_bin': ['25 or under', '26-33', '34+'],
           'payment_bin': ['up to 80', '81-150', 'over 150'], 'credit_bin': ['up to 1,500', '1,501-4,000', '4,001-7,500', 'over 7,500']}
rows = []
for f in final:
    t = final_table[final_table['Feature'] == f].set_index('Band')
    pattern = '- (not ordered)'
    if f in ordered:
        w = t.loc[ordered[f], 'WoE']
        pattern = 'Monotonic' if (w.is_monotonic_increasing or w.is_monotonic_decreasing) else 'Not monotonic'
    rows.append([names[f], len(t), round(t['Customers'].min() / 10, 1), int(t['Good'].min()), int(t['Bad'].min()),
                 round(t['IV part'].sum(), 4), round(woe_iv(fe, f)[1], 4), pattern])
woe_review = pd.DataFrame(rows, columns=['Feature', 'Bands', 'Smallest band (%)', 'Fewest good', 'Fewest bad', 'IV (saved)', 'IV (recalculated)', 'Pattern'])
woe_review

,Feature,Bands,Smallest band (%),Fewest good,Fewest bad,IV (saved),IV (recalculated),Pattern
0,Checking account status,4,6.3,49,14,0.6621,0.6621,- (not ordered)
1,Duration,4,17.0,88,27,0.2301,0.2301,Monotonic
2,Saving account status,4,10.3,69,17,0.1885,0.1885,- (not ordered)
3,Credit amount,4,8.6,42,44,0.1423,0.1422,Not monotonic
4,Monthly payment,3,19.7,115,82,0.0771,0.0771,Monotonic
5,Age,3,19.0,110,80,0.0929,0.0929,Monotonic
6,Housing (own vs rent/free),2,28.7,173,114,0.0832,0.0831,- (not ordered)
7,Purpose (radio/TV vs other),2,28.0,218,62,0.0571,0.0571,- (not ordered)


All bands meet the rules (smallest band 6.3%). Duration, age and monthly payment are monotonic. Credit amount is the only
non-monotonic feature, because the smallest loans have a slightly higher bad rate than mid-sized loans; notebook 08 showed this small
difference is **not** statistically significant, so it does not affect the scorecard's logic (see the band-merging test in Section 3.5).

## 2. Validation of the feature calculations

### 2.1 Rule-by-rule recalculation
Each band is recalculated with plain if / else rules (independent of `pd.cut`) and compared with the saved bands.

In [3]:
def duration_band(x): return 'up to 11' if x <= 11 else '12-23' if x <= 23 else '24-35' if x <= 35 else '36+'
def age_band(x): return '25 or under' if x <= 25 else '26-33' if x <= 33 else '34+'
def credit_band(x): return 'up to 1,500' if x <= 1500 else '1,501-4,000' if x <= 4000 else '4,001-7,500' if x <= 7500 else 'over 7,500'
def payment_band(x): return 'up to 80' if x <= 80 else '81-150' if x <= 150 else 'over 150'

rule_check = pd.DataFrame({
    'Feature': ['Duration', 'Age', 'Credit amount', 'Monthly payment', 'Monthly payment value', 'Housing (own vs rent/free)', 'Purpose (radio/TV vs other)'],
    'Matches saved values': [
        (raw['Duration'].map(duration_band) == fe['duration_bin']).all(),
        (raw['Age'].map(age_band) == fe['age_bin']).all(),
        (raw['Credit amount'].map(credit_band) == fe['credit_bin']).all(),
        ((raw['Credit amount'] / raw['Duration']).map(payment_band) == fe['payment_bin']).all(),
        np.allclose(raw['Credit amount'] / raw['Duration'], fe['monthly_payment'], atol=0.005),
        (np.where(raw['Housing'] == 'own', 'own', 'rent/free') == fe['home_owner']).all(),
        (np.where(raw['Purpose'] == 'radio/TV', 'radio/TV', 'other') == fe['purpose_type']).all(),
    ]})
rule_check

,Feature,Matches saved values
0,Duration,True
1,Age,True
2,Credit amount,True
3,Monthly payment,True
4,Monthly payment value,True
5,Housing (own vs rent/free),True
6,Purpose (radio/TV vs other),True


### 2.2 Boundary tests
Values exactly on and just past every cut point must fall in the right band.

In [4]:
boundary_cases = [
    ('Duration', 11, 'up to 11'), ('Duration', 12, '12-23'), ('Duration', 23, '12-23'), ('Duration', 24, '24-35'),
    ('Duration', 35, '24-35'), ('Duration', 36, '36+'),
    ('Age', 25, '25 or under'), ('Age', 26, '26-33'), ('Age', 33, '26-33'), ('Age', 34, '34+'),
    ('Credit amount', 1500, 'up to 1,500'), ('Credit amount', 1501, '1,501-4,000'), ('Credit amount', 4000, '1,501-4,000'),
    ('Credit amount', 4001, '4,001-7,500'), ('Credit amount', 7500, '4,001-7,500'), ('Credit amount', 7501, 'over 7,500'),
    ('Monthly payment', 80, 'up to 80'), ('Monthly payment', 80.01, '81-150'), ('Monthly payment', 150, '81-150'), ('Monthly payment', 150.01, 'over 150'),
]
cut_rules = {'Duration': ([0, 11, 23, 35, 100], ['up to 11', '12-23', '24-35', '36+']), 'Age': ([18, 25, 33, 100], ['25 or under', '26-33', '34+']),
             'Credit amount': ([0, 1500, 4000, 7500, 1e9], ['up to 1,500', '1,501-4,000', '4,001-7,500', 'over 7,500']),
             'Monthly payment': ([0, 80, 150, 1e9], ['up to 80', '81-150', 'over 150'])}
rows = []
for feature, value, expected in boundary_cases:
    edges, labels = cut_rules[feature]
    got = str(pd.cut(pd.Series([value]), edges, labels=labels)[0])
    rows.append([feature, value, expected, got, got == expected])
boundary = pd.DataFrame(rows, columns=['Feature', 'Value', 'Expected band', 'Band given', 'Correct'])
print('All boundary cases correct:', boundary['Correct'].all())
boundary

All boundary cases correct: True


,Feature,Value,Expected band,Band given,Correct
0,Duration,11.00,up to 11,up to 11,True
1,Duration,12.00,12-23,12-23,True
2,Duration,23.00,12-23,12-23,True
3,Duration,24.00,24-35,24-35,True
4,Duration,35.00,24-35,24-35,True
5,Duration,36.00,36+,36+,True
6,Age,25.00,25 or under,25 or under,True
7,Age,26.00,26-33,26-33,True
8,Age,33.00,26-33,26-33,True
9,Age,34.00,34+,34+,True


In [5]:
rounding_effect = (pd.cut(raw['Credit amount'] / raw['Duration'], [0, 80, 150, 1e9]).astype(str).values
                   != pd.cut(fe['monthly_payment'], [0, 80, 150, 1e9]).astype(str).values).sum()
print('Customers whose payment band changes because monthly payment is rounded to 2 decimals:', rounding_effect)

mismatch = 0
for f in final:
    mapping = final_table[final_table['Feature'] == f].set_index('Band')['WoE']
    mismatch += int((fe[f].map(mapping).round(4) != final_woe['woe_' + f].round(4)).sum())
print('Rows where the final WoE dataset disagrees with the final WoE table:', mismatch)

Customers whose payment band changes because monthly payment is rounded to 2 decimals: 0
Rows where the final WoE dataset disagrees with the final WoE table: 0


## 3. Check of the feature-selection methodology

### 3.1 The selection rules being checked
1. Keep features with **IV of at least 0.02**.
2. Exclude **protected characteristics** (sex).
3. Among features correlated above **0.6** (on the WoE scale), keep only the one with the higher IV.

### 3.2 Is the selection stable and unbiased? (nested cross-validation)
If features are chosen using all the data and then tested on the same data, the result can be too optimistic.
Here the **whole process** (WoE, IV, rules 1 to 3, model) is repeated inside each training fold of a 5-fold split,
repeated with 5 different seeds (25 folds), and scored only on the held-out fold.

In [6]:
candidates = ['checking_status', 'low_balance_profile', 'duration_bin', 'saving_status', 'credit_bin', 'age_bin',
              'home_owner', 'payment_bin', 'purpose_type', 'sex', 'job_group']

def select(data, iv_cut=0.02, corr_cut=0.6):
    ivs = {c: woe_iv(data, c)[1] for c in candidates}
    X = pd.DataFrame({c: data[c].map(woe_iv(data, c)[0]) for c in candidates})
    chosen = []
    for c in sorted(candidates, key=lambda c: -ivs[c]):
        if ivs[c] < iv_cut or c == 'sex':
            continue
        if any(abs(X[c].corr(X[s])) > corr_cut for s in chosen):
            continue
        chosen.append(c)
    return chosen

def fit_score(train, test, feats):
    X_tr = pd.DataFrame({f: train[f].map(woe_iv(train, f)[0]) for f in feats})
    X_te = pd.DataFrame({f: test[f].map(woe_iv(train, f)[0]).fillna(0) for f in feats})
    return roc_auc_score(test['bad'], LogisticRegression(max_iter=1000).fit(X_tr, train['bad']).predict_proba(X_te)[:, 1])

times_selected = {c: 0 for c in candidates}
nested_auc, fixed_auc = [], []
for seed in [1, 2, 3, 4, 5]:
    for tr_idx, te_idx in StratifiedKFold(n_splits=5, shuffle=True, random_state=seed).split(fe, fe['bad']):
        train, test = fe.iloc[tr_idx], fe.iloc[te_idx]
        chosen = select(train)
        for c in chosen:
            times_selected[c] += 1
        nested_auc.append(fit_score(train, test, chosen))
        fixed_auc.append(fit_score(train, test, final))

X_all = pd.DataFrame({f: fe[f].map(woe_iv(fe, f)[0]) for f in final})
in_sample = roc_auc_score(fe['bad'], LogisticRegression(max_iter=1000).fit(X_all, fe['bad']).predict_proba(X_all)[:, 1])

selection_frequency = pd.DataFrame({'Feature': [names[c] for c in candidates],
                                    'Selected in (% of 25 folds)': [times_selected[c] * 4 for c in candidates],
                                    'In final set': ['Yes' if c in final else 'No' for c in candidates]})
optimism = pd.DataFrame({'Estimate': ['In-sample (same data used to choose and test)', 'Fixed final features, cross-validated',
                                      'Full selection repeated inside each fold (nested)'],
                         'ROC-AUC': [round(in_sample, 4), round(np.mean(fixed_auc), 4), round(np.mean(nested_auc), 4)]})
print(optimism)
selection_frequency

                                            Estimate  ROC-AUC
0      In-sample (same data used to choose and test)   0.7908
1              Fixed final features, cross-validated   0.7768
2  Full selection repeated inside each fold (nested)   0.7764


,Feature,Selected in (% of 25 folds),In final set
0,Checking account status,100,Yes
1,Low balance profile,8,No
2,Duration,100,Yes
3,Saving account status,100,Yes
4,Credit amount,100,Yes
5,Age,100,Yes
6,Housing (own vs rent/free),100,Yes
7,Monthly payment,100,Yes
8,Purpose (radio/TV vs other),100,Yes
9,Sex,0,No


- All 8 final features are chosen in **every** fold, and the excluded features are almost never chosen, so the selection is **stable**.
- The nested ROC-AUC is practically the same as the cross-validated ROC-AUC of the fixed final set, so choosing the features on the
  full data has **not** made the results look better than they are. Only the in-sample figure is optimistic, which is why it is never used for reporting.

### 3.3 Do other selection methods agree?
The final set is compared with four independent methods applied to all 10 non-protected candidates:
**L1 (lasso) logistic regression**, **backward elimination by AIC**, **mutual information** and **random forest permutation importance**
(on the raw, un-binned data).

In [7]:
cand10 = [c for c in candidates if c != 'sex']
X10 = pd.DataFrame({c: fe[c].map(woe_iv(fe, c)[0]) for c in cand10})

lasso = LogisticRegressionCV(Cs=20, penalty='l1', solver='liblinear', cv=5, scoring='roc_auc', random_state=0).fit(X10, fe['bad'])
lasso_coef = pd.Series(lasso.coef_[0], index=cand10)

kept = list(cand10)
while True:
    base_aic = sm.Logit(fe['bad'], sm.add_constant(X10[kept])).fit(disp=0).aic
    trials = {c: sm.Logit(fe['bad'], sm.add_constant(X10[[k for k in kept if k != c]])).fit(disp=0).aic for c in kept}
    worst = min(trials, key=trials.get)
    if trials[worst] >= base_aic:
        break
    kept.remove(worst)

mutual_info = pd.Series(mutual_info_classif(X10, fe['bad'], discrete_features=True, random_state=0), index=cand10)

raw_X = raw.drop(columns='Risk').copy()
raw_X[['Saving accounts', 'Checking account']] = raw_X[['Saving accounts', 'Checking account']].fillna('missing')
raw_X['Monthly payment'] = raw_X['Credit amount'] / raw_X['Duration']
raw_X = pd.get_dummies(raw_X, columns=['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose'], dtype=float)
forest = RandomForestClassifier(n_estimators=500, min_samples_leaf=10, random_state=0, n_jobs=-1).fit(raw_X, fe['bad'])
perm = permutation_importance(forest, raw_X, fe['bad'], n_repeats=10, random_state=0, scoring='roc_auc')
perm = pd.Series(perm.importances_mean, index=raw_X.columns)
raw_group = {'Checking account': 'checking_status', 'Duration': 'duration_bin', 'Saving accounts': 'saving_status',
             'Credit amount': 'credit_bin', 'Monthly payment': 'payment_bin', 'Age': 'age_bin', 'Housing': 'home_owner',
             'Purpose': 'purpose_type', 'Job': 'job_group', 'Sex': 'sex'}
forest_imp = perm.groupby(lambda c: next(k for k in raw_group if c.startswith(k))).sum().rename(index=raw_group)

ivs10 = pd.Series({c: woe_iv(fe, c)[1] for c in cand10})
agreement = pd.DataFrame({
    'Feature': [names[c] for c in cand10],
    'IV': ivs10.round(3).values,
    'Lasso coefficient': lasso_coef.round(3).values,
    'Kept by backward AIC': ['Yes' if c in kept else 'No' for c in cand10],
    'Mutual information': mutual_info.round(4).values,
    'Forest importance': [round(forest_imp[c], 4) if c in forest_imp else '- (not a raw column)' for c in cand10],
    'In final set': ['Yes' if c in final else 'No' for c in cand10],
})
print('Rank agreement between IV and mutual information (Spearman):', round(spearmanr(ivs10, mutual_info)[0], 3))
agreement

Rank agreement between IV and mutual information (Spearman): 1.0


,Feature,IV,Lasso coefficient,Kept by backward AIC,Mutual information,Forest importance,In final set
0,Checking account status,0.662,-0.813,Yes,0.0657,0.1022,Yes
1,Low balance profile,0.327,-0.066,No,0.0347,- (not a raw column),No
2,Duration,0.230,-0.616,Yes,0.0237,0.038,Yes
3,Saving account status,0.189,-0.668,Yes,0.0192,0.0248,Yes
4,Credit amount,0.142,-0.903,Yes,0.0151,0.038,Yes
5,Age,0.093,-0.662,Yes,0.0098,0.026,Yes
6,Housing (own vs rent/free),0.083,-0.502,Yes,0.0088,0.0102,Yes
7,Monthly payment,0.077,-1.142,Yes,0.0082,0.0511,Yes
8,Purpose (radio/TV vs other),0.057,-0.563,Yes,0.0059,0.0187,Yes
9,Job group,0.008,0.000,No,0.0009,0.0079,No


- **Backward AIC** keeps exactly the 8 final features.
- **Lasso** sets Job group to zero and gives Low balance profile an almost-zero weight, confirming both exclusions.
- **Mutual information** ranks the features in almost the same order as IV.
- **Random forest**, which uses the raw data and no WoE at all, also ranks Checking account first and Job group, Housing and Sex last.

### 3.4 Are the results sensitive to the thresholds?
The selection is repeated for different IV cut-offs and correlation limits.

In [8]:
rows = []
for iv_cut in [0.02, 0.05, 0.10]:
    for corr_cut in [0.5, 0.6, 0.7, 0.8]:
        chosen = select(fe, iv_cut, corr_cut)
        rows.append([iv_cut, corr_cut, len(chosen), 'Yes' if sorted(chosen) == sorted(final) else 'No',
                     ', '.join(names[c] for c in chosen if c not in final) or '-',
                     ', '.join(names[c] for c in final if c not in chosen) or '-'])
sensitivity = pd.DataFrame(rows, columns=['IV cut-off', 'Correlation limit', 'Features', 'Same as final', 'Added', 'Removed'])
sensitivity

,IV cut-off,Correlation limit,Features,Same as final,Added,Removed
0,0.02,0.5,8,Yes,-,-
1,0.02,0.6,8,Yes,-,-
2,0.02,0.7,9,No,Low balance profile,-
3,0.02,0.8,9,No,Low balance profile,-
4,0.05,0.5,8,Yes,-,-
5,0.05,0.6,8,Yes,-,-
6,0.05,0.7,9,No,Low balance profile,-
7,0.05,0.8,9,No,Low balance profile,-
8,0.10,0.5,4,No,-,"Monthly payment, Age, Housing (own vs rent/free), Purpose (radio/TV vs other)"
9,0.10,0.6,4,No,-,"Monthly payment, Age, Housing (own vs rent/free), Purpose (radio/TV vs other)"


The final set is unchanged for any IV cut-off between 0.02 and 0.05 and any correlation limit between 0.5 and 0.6.
A looser correlation limit (0.7 or more) would add Low balance profile, which the lasso and AIC tests show adds nothing.
A strict IV cut-off of 0.10 would drop the four Tier 3 / weaker features, which the earlier hold-out tests showed do add predictive power.

### 3.5 Are the bands too fine?
Notebook 08 found some neighbouring bands that do not differ significantly. Here they are merged and the result is compared over 100 random splits.

In [9]:
fe['credit_merged'] = fe['credit_bin'].replace({'up to 1,500': 'up to 4,000', '1,501-4,000': 'up to 4,000'})
fe['duration_merged'] = fe['duration_bin'].replace({'12-23': '12-35', '24-35': '12-35'})
fe['saving_merged'] = fe['saving_status'].replace({'little': 'little/moderate', 'moderate': 'little/moderate'})
fe['payment_merged'] = fe['payment_bin'].replace({'81-150': 'over 80', 'over 150': 'over 80'})
merged = ['checking_status', 'duration_merged', 'saving_merged', 'credit_merged', 'payment_merged', 'age_bin', 'home_owner', 'purpose_type']

auc_final, auc_merged = [], []
for tr_idx, te_idx in StratifiedShuffleSplit(n_splits=100, test_size=0.3, random_state=11).split(fe, fe['bad']):
    train, test = fe.iloc[tr_idx], fe.iloc[te_idx]
    auc_final.append(fit_score(train, test, final))
    auc_merged.append(fit_score(train, test, merged))
band_merge = pd.DataFrame({'Version': ['Final bands', 'Merged bands (fewer, all monotonic)'],
                           'Total bands': [sum(fe[f].nunique() for f in final), sum(fe[f].nunique() for f in merged)],
                           'Mean hold-out ROC-AUC': [round(np.mean(auc_final), 4), round(np.mean(auc_merged), 4)]})
print('Merged version better in', round(np.mean(np.array(auc_merged) > np.array(auc_final)) * 100), '% of splits')
band_merge

Merged version better in 52 % of splits


,Version,Total bands,Mean hold-out ROC-AUC
0,Final bands,26,0.7745
1,"Merged bands (fewer, all monotonic)",22,0.7755


The merged version performs the same (a difference of about 0.001, better in only about half of the splits), so the final bands are
**not over-fitted**. The final bands are kept because they are already validated; the merged version is a valid option if an even
simpler scorecard is wanted.

### 3.6 Leakage review
| Question | Answer |
|---|---|
| Are all features known at the time of the application? | Yes: account balances, loan amount, duration, purpose, age, housing and job are all application details. |
| Is the target used to build any feature? | Only through WoE. In every validation, WoE was learned on the training part only and applied to the test part. |
| Were the cut points chosen with the target? | Yes, on the full data in notebook 05. The nested test in 3.2 and the band-merging test in 3.5 show this did not inflate performance. |

### 3.7 Fairness check: do the final features act as a stand-in for sex?

In [10]:
rows = []
for f in final:
    table = pd.crosstab(fe[f], fe['sex'])
    chi2 = chi2_contingency(table)[0]
    rows.append([names[f], round(np.sqrt(chi2 / (len(fe) * (min(table.shape) - 1))), 3)])
proxy = pd.DataFrame(rows, columns=['Feature', "Association with sex (Cramer's V)"]).sort_values("Association with sex (Cramer's V)", ascending=False)
sex_auc = cross_val_score(LogisticRegression(max_iter=2000), pd.get_dummies(fe[final], dtype=float),
                          (fe['sex'] == 'female').astype(int), cv=5, scoring='roc_auc').mean()
print('How well the 8 final features together predict sex (ROC-AUC, 0.5 = not at all):', round(sex_auc, 3))
proxy

How well the 8 final features together predict sex (ROC-AUC, 0.5 = not at all): 0.637


,Feature,Association with sex (Cramer's V)
5,Age,0.268
3,Credit amount,0.120
6,Housing (own vs rent/free),0.117
1,Duration,0.099
4,Monthly payment,0.054
2,Saving account status,0.044
0,Checking account status,0.027
7,Purpose (radio/TV vs other),0.006


No feature is a strong stand-in for sex. Age has the largest link (moderate, because women in the data are younger on average),
and together the features predict sex only weakly (ROC-AUC about 0.64). Age is itself a protected characteristic in some jurisdictions,
so its use should be confirmed against local lending rules.

## 4. Methodology verdict

In [11]:
verdict = pd.DataFrame([
    ['WoE / IV of the final bands', 'Sound', 'All bands meet the 5% and both-classes rules; saved IVs match a fresh calculation'],
    ['Feature calculations', 'Correct', 'Rule-by-rule recalculation and all 20 boundary tests pass; no rounding effects'],
    ['Stability of the selection', 'Stable', 'All 8 final features chosen in 100% of 25 nested folds'],
    ['Optimism from selecting on the full data', 'None found', f'Nested ROC-AUC {np.mean(nested_auc):.3f} vs fixed-set ROC-AUC {np.mean(fixed_auc):.3f}'],
    ['Agreement with other methods', 'Strong', 'Backward AIC keeps the same 8; lasso and random forest agree on the exclusions'],
    ['Sensitivity to thresholds', 'Robust', 'Same 8 features for IV cut-offs 0.02 to 0.05 and correlation limits 0.5 to 0.6'],
    ['Granularity of the bands', 'Appropriate', 'Merging indistinct bands changes ROC-AUC by about 0.001'],
    ['Leakage', 'None found', 'Application-time features only; WoE learned within training folds'],
    ['Fairness', 'Acceptable, with one action', 'Sex excluded; no strong proxy; age use to be confirmed against local rules'],
], columns=['Aspect', 'Verdict', 'Evidence'])
verdict

,Aspect,Verdict,Evidence
0,WoE / IV of the final bands,Sound,All bands meet the 5% and both-classes rules; saved IVs match a fresh calculation
1,Feature calculations,Correct,Rule-by-rule recalculation and all 20 boundary tests pass; no rounding effects
2,Stability of the selection,Stable,All 8 final features chosen in 100% of 25 nested folds
3,Optimism from selecting on the full data,None found,Nested ROC-AUC 0.776 vs fixed-set ROC-AUC 0.777
4,Agreement with other methods,Strong,Backward AIC keeps the same 8; lasso and random forest agree on the exclusions
5,Sensitivity to thresholds,Robust,Same 8 features for IV cut-offs 0.02 to 0.05 and correlation limits 0.5 to 0.6
6,Granularity of the bands,Appropriate,Merging indistinct bands changes ROC-AUC by about 0.001
7,Leakage,None found,Application-time features only; WoE learned within training folds
8,Fairness,"Acceptable, with one action",Sex excluded; no strong proxy; age use to be confirmed against local rules


## 5. Quick checks

In [12]:
checks = {
    'Every final band holds at least 5% of customers': (woe_review['Smallest band (%)'] >= 5).all(),
    'Every final band has good and bad customers': (woe_review['Fewest good'] > 0).all() and (woe_review['Fewest bad'] > 0).all(),
    'Saved IVs match a fresh calculation': (abs(woe_review['IV (saved)'] - woe_review['IV (recalculated)']) < 0.001).all(),
    'Rule-by-rule recalculation matches all features': rule_check['Matches saved values'].all(),
    'All 20 boundary cases fall in the correct band': boundary['Correct'].all() and len(boundary) == 20,
    'Final WoE dataset matches the final WoE table': mismatch == 0,
    'All 8 final features selected in every nested fold': all(times_selected[c] == 25 for c in final),
    'Nested ROC-AUC within 0.01 of the fixed-set ROC-AUC': abs(np.mean(nested_auc) - np.mean(fixed_auc)) < 0.01,
    'Backward AIC keeps exactly the final features': sorted(kept) == sorted(final),
    'Lasso gives Job group a zero weight': lasso_coef['job_group'] == 0,
    'Final set unchanged for IV cut-off 0.02-0.05 and correlation limit 0.5-0.6':
        sensitivity[(sensitivity['IV cut-off'] <= 0.05) & (sensitivity['Correlation limit'] <= 0.6)]['Same as final'].eq('Yes').all(),
    'Sex is not in the final set': 'sex' not in final,
}
for name, passed in checks.items():
    print('PASS' if passed else 'FAIL', '-', name)
assert all(checks.values())
print()
print(f'All {len(checks)} checks passed')

PASS - Every final band holds at least 5% of customers
PASS - Every final band has good and bad customers
PASS - Saved IVs match a fresh calculation
PASS - Rule-by-rule recalculation matches all features
PASS - All 20 boundary cases fall in the correct band
PASS - Final WoE dataset matches the final WoE table
PASS - All 8 final features selected in every nested fold
PASS - Nested ROC-AUC within 0.01 of the fixed-set ROC-AUC
PASS - Backward AIC keeps exactly the final features
PASS - Lasso gives Job group a zero weight
PASS - Final set unchanged for IV cut-off 0.02-0.05 and correlation limit 0.5-0.6
PASS - Sex is not in the final set

All 12 checks passed


## 6. Final recommendations

1. **Approve the 8 final features** for the scorecard: Checking account status, Duration, Saving account status, Credit amount,
   Monthly payment, Age, Housing (own vs rent/free) and Purpose (radio/TV vs other).
2. **Use the final files**: `german_credit_woe_final.csv` for modelling and `final_woe_iv_table.csv` for the scorecard points.
3. **Learn WoE on the training data only** when the final model is fitted, exactly as in every validation here.
4. **Keep the current bands.** Merging further gives no gain; the merged version is an option only if a smaller scorecard is wanted.
5. **Keep Sex, Job group and Low balance profile out**, for the reasons confirmed above.
6. **Confirm the use of Age** with the compliance or risk team, as age is protected in some jurisdictions.
7. **Confirm with the data owner what a missing checking account means**, because it drives the strongest feature.
8. **Monitor all 8 features after deployment** with the Population Stability Index, and re-run notebooks 07 to 09 when new data arrives.